# Rotation prediction SSL pretraining, ps32

This notebook pretrains a modified 14-channel ResNet-18 encoder using a rotation prediction self-supervised task on unlabeled conditioning-factor raster patches.

## 1. Purpose and experiment configuration

This notebook only performs Rotation SSL pretraining. It does not use landslide/non-landslide labels, does not fine-tune a supervised classifier, and does not generate susceptibility maps.

This is in-domain transductive SSL pretraining: unlabeled patches are sampled from the whole cleaned study area. SCV holdout clusters are not excluded because no supervised labels are used during pretraining.

Scientific caution: Rotation prediction is included as a simple geometric SSL baseline. Because geographic orientation may have physical meaning in landslide susceptibility modeling, especially for terrain-related variables, rotation prediction should not be interpreted as a physically invariant transformation. It is evaluated here as an empirical pretext-task ablation.

Current design note: this first run does not implement factor-specific physical transformations for aspect, flow direction, or other directional variables. All channels are treated uniformly for consistency with the current SSL comparison design.

In [1]:
from pathlib import Path
import os
import numpy as np

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

raster_dir = PROJECT_ROOT / "data/processed/rasters_cleaned"
unlabeled_index_csv = PROJECT_ROOT / "data/processed/ssl_unlabeled_indices/unlabeled_patch_index_ps32_n20000.csv"
output_root = PROJECT_ROOT / "outputs/SSL_rotation_ps32"
training_log_dir = output_root / "training_logs"
figure_root = PROJECT_ROOT / "figures/SSL_rotation_ps32"
training_curve_dir = figure_root / "training_curves"
rotation_example_dir = figure_root / "rotation_examples"
confusion_dir = figure_root / "confusion"
checkpoint_dir = PROJECT_ROOT / "checkpoints/ssl_pretrained/rotation"

training_log_path = training_log_dir / "rotation_ps32_training_log.csv"
encoder_best_path = checkpoint_dir / "resnet18_rotation_ps32_encoder_best.pt"
full_model_best_path = checkpoint_dir / "resnet18_rotation_ps32_full_model_best.pt"
last_checkpoint_path = checkpoint_dir / "resnet18_rotation_ps32_last.pt"
loss_acc_curve_png = training_curve_dir / "rotation_ps32_loss_accuracy_curve.png"
loss_acc_curve_pdf = training_curve_dir / "rotation_ps32_loss_accuracy_curve.pdf"
rotation_example_png = rotation_example_dir / "rotation_examples_ps32.png"
rotation_example_pdf = rotation_example_dir / "rotation_examples_ps32.pdf"
confusion_png = confusion_dir / "rotation_val_confusion_ps32.png"
confusion_pdf = confusion_dir / "rotation_val_confusion_ps32.pdf"

patch_size = 32
n_rotation_classes = 4
n_unlabeled_patches = 20_000
normalization_sample_size = 5_000
max_nodata_ratio = 0.0
nodata_value = -9999
random_seed = 42
max_attempts = 1_000_000
train_fraction = 0.9
batch_size = 128
num_workers = 0
learning_rate = 1e-4
weight_decay = 1e-4
max_epochs = 50
early_stopping_patience = 10
gradient_clip_norm = 5.0
random_ce_baseline = float(np.log(n_rotation_classes))
random_acc_baseline = 1.0 / n_rotation_classes

print("Project root:", PROJECT_ROOT)
print("Raster dir:", raster_dir)
print("Unlabeled index:", unlabeled_index_csv)
print("Encoder checkpoint output:", encoder_best_path)

Project root: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project
Raster dir: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\data\processed\rasters_cleaned
Unlabeled index: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\data\processed\ssl_unlabeled_indices\unlabeled_patch_index_ps32_n20000.csv
Encoder checkpoint output: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\rotation\resnet18_rotation_ps32_encoder_best.pt


## 2. Import packages and project modules

In [2]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

from src.patch_dataset import list_raster_files, audit_raster_alignment
from src.ssl_masked_recon import MaskedReconstructionRasterDataset, compute_ssl_channel_stats, create_unlabeled_patch_index
from src.ssl_rotation import RotationRasterPatchDataset, RotationResNet18Model, apply_rotation, ROTATION_CLASS_LABELS
from src.train_ssl import evaluate_rotation, train_rotation_model
from src.plotting import (
    plot_rotation_confusion_matrix,
    plot_rotation_examples,
    plot_rotation_training_curves,
    set_publication_plot_style,
)
from src.utils import count_trainable_parameters, ensure_dir, set_global_seed

for directory in [unlabeled_index_csv.parent, training_log_dir, training_curve_dir, rotation_example_dir, confusion_dir, checkpoint_dir]:
    ensure_dir(directory)

## 3. Set random seeds and device

In [3]:
set_global_seed(random_seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = bool(torch.cuda.is_available())
print("Using device:", device)
if torch.cuda.is_available():
    print("CUDA GPU:", torch.cuda.get_device_name(0))
    print("torch.version.cuda:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available; Rotation pretraining will run on CPU.")

Using device: cuda
CUDA GPU: NVIDIA GeForce RTX 4090
torch.version.cuda: 11.8


## 4. Set publication-quality plotting style

In [4]:
plot_font_family = set_publication_plot_style(font_family="Arial", font_size=10)
print("Plotting font family:", plot_font_family)
print("Plotting font size:", 10)

Plotting font family: Arial
Plotting font size: 10


## 5. Load and audit cleaned rasters

In [5]:
raster_files = list_raster_files(raster_dir)
raster_audit = audit_raster_alignment(raster_files, expected_nodata=nodata_value)
print("number of raster files:", len(raster_files))
print("raster shape:", (raster_audit.height, raster_audit.width))
print("raster CRS:", raster_audit.crs)
print("raster nodata:", raster_audit.nodata)
print("patch_size:", patch_size)
print("n_rotation_classes:", n_rotation_classes)

number of raster files: 14
raster shape: (10071, 9596)
raster CRS: EPSG:32618
raster nodata: -9999.0
patch_size: 32
n_rotation_classes: 4


## 6. Load or generate unlabeled patch index

In [6]:
unlabeled_index = create_unlabeled_patch_index(
    raster_dir=raster_dir,
    output_csv=unlabeled_index_csv,
    patch_size=patch_size,
    n_patches=n_unlabeled_patches,
    nodata_value=nodata_value,
    max_nodata_ratio=max_nodata_ratio,
    random_seed=random_seed,
    max_attempts=max_attempts,
    force_regenerate=False,
)
valid_unlabeled = unlabeled_index.loc[unlabeled_index["valid_patch"].astype(bool)].copy()
if len(valid_unlabeled) < n_unlabeled_patches:
    raise RuntimeError(f"Expected at least {n_unlabeled_patches} valid unlabeled patches, found {len(valid_unlabeled)}.")
valid_unlabeled = valid_unlabeled.iloc[:n_unlabeled_patches].copy()
if not np.isclose(valid_unlabeled["nodata_ratio"].to_numpy(dtype="float64"), 0.0).all():
    raise ValueError("Unlabeled patch index contains nonzero nodata_ratio rows.")
print("number of valid unlabeled patches:", len(valid_unlabeled))
print(valid_unlabeled[["patch_id", "row", "col", "nodata_ratio"]].head().to_string(index=False))

number of valid unlabeled patches: 20000
    patch_id  row  col  nodata_ratio
U_SSL_000001 2038  916           0.0
U_SSL_000002 5168 1241           0.0
U_SSL_000003 5491 4256           0.0
U_SSL_000004 4538 2189           0.0
U_SSL_000005  940 5320           0.0


## 7. Compute Rotation SSL channel normalization statistics

In [7]:
raw_stats_dataset = MaskedReconstructionRasterDataset(
    patch_index_csv=unlabeled_index_csv,
    raster_dir=raster_dir,
    patch_size=patch_size,
    nodata_value=nodata_value,
    normalize=False,
    return_metadata=False,
)
channel_means, channel_stds = compute_ssl_channel_stats(
    raw_stats_dataset,
    sample_size=normalization_sample_size,
    batch_size=64,
    random_seed=random_seed,
)
raw_stats_dataset.close()
print("Channel means shape:", channel_means.shape)
print("Channel stds shape:", channel_stds.shape)
print("Channel means:", np.round(channel_means, 4))
print("Channel stds:", np.round(channel_stds, 4))
if not np.isfinite(channel_means).all() or not np.isfinite(channel_stds).all():
    raise ValueError("Channel normalization statistics contain NaN or inf.")
if (channel_stds <= 0).any():
    raise ValueError("At least one channel std is nonpositive.")

Channel means shape: (14,)
Channel stds shape: (14,)
Channel means: [1.8172540e+02 1.5900377e+03 1.9745800e+01 1.7168539e+02 1.9745800e+01
 4.2806499e+01 1.0002100e+01 6.5240002e-01 1.7899999e-02 1.7500000e-02
 5.2075802e+01 5.5462999e+00 2.4302999e+01 7.0296001e+00]
Channel stds: [7.987370e+01 6.338770e+01 3.289600e+00 1.632223e+02 3.289600e+00
 2.239570e+01 3.565600e+00 1.664000e-01 1.545000e-01 2.632000e-01
 9.396800e+00 5.474800e+00 1.303064e+02 1.296800e+00]


## 8. Define Rotation Dataset and DataLoaders

In [8]:
rotation_dataset = RotationRasterPatchDataset(
    patch_index_csv=unlabeled_index_csv,
    raster_dir=raster_dir,
    patch_size=patch_size,
    nodata_value=nodata_value,
    normalize=True,
    channel_means=channel_means,
    channel_stds=channel_stds,
    return_metadata=False,
    random_seed=random_seed,
)

X_rot, y_rot = rotation_dataset[0]
print("Dataset length:", len(rotation_dataset))
print("X_rot.shape:", tuple(X_rot.shape))
print("y_rot:", int(y_rot))
print("X_rot finite:", bool(torch.isfinite(X_rot).all()))
print("X_rot contains nodata:", bool((X_rot == nodata_value).any()))
if X_rot.shape != (14, patch_size, patch_size):
    raise ValueError(X_rot.shape)
if not (0 <= int(y_rot) < n_rotation_classes):
    raise ValueError(int(y_rot))
print("Rotation transformation preserves shape and channel count.")

rng = np.random.default_rng(random_seed)
all_indices = np.arange(len(rotation_dataset))
rng.shuffle(all_indices)
n_train = int(train_fraction * len(all_indices))
train_indices = all_indices[:n_train].tolist()
val_indices = all_indices[n_train:].tolist()
train_dataset = Subset(rotation_dataset, train_indices)
val_dataset = Subset(rotation_dataset, val_indices)

loader_generator = torch.Generator()
loader_generator.manual_seed(random_seed)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory, generator=loader_generator)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
print("Train/val split sizes:", len(train_dataset), len(val_dataset))
print("batch size:", batch_size)
print("pin_memory:", pin_memory)

Dataset length: 20000
X_rot.shape: (14, 32, 32)
y_rot: 0
X_rot finite: True
X_rot contains nodata: False
Rotation transformation preserves shape and channel count.
Train/val split sizes: 18000 2000
batch size: 128
pin_memory: True


## 9. Define modified ResNet-18 encoder and Rotation classification head

In [9]:
model = RotationResNet18Model(
    in_channels=14,
    n_rotation_classes=n_rotation_classes,
    small_patch_stem=True,
    dropout=0.3,
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
print("model parameter count:", count_trainable_parameters(model))
print("n_rotation_classes:", n_rotation_classes)

model parameter count: 11177220
n_rotation_classes: 4


## 10. Define Rotation training objective and accuracy metrics

In [10]:
print("Random CE baseline ln(4):", random_ce_baseline)
print("Random top-1 accuracy baseline:", random_acc_baseline)
smoke_batch = next(iter(train_loader))
smoke_X = smoke_batch[0].to(device, non_blocking=True)
smoke_y = smoke_batch[1].to(device, non_blocking=True)
print("model device:", next(model.parameters()).device)
print("X_rot.device:", smoke_X.device)
print("y_rot.device:", smoke_y.device)
with torch.no_grad():
    smoke_logits = model(smoke_X[:4])
print("logits shape:", tuple(smoke_logits.shape))
if torch.cuda.is_available():
    assert next(model.parameters()).is_cuda and smoke_X.is_cuda and smoke_y.is_cuda

Random CE baseline ln(4): 1.3862943611198906
Random top-1 accuracy baseline: 0.25


model device: cuda:0
X_rot.device: cuda:0
y_rot.device: cuda:0
logits shape: (4, 4)


## 11. Train Rotation SSL model

In [11]:
config = {
    "ssl_task": "rotation_prediction",
    "patch_size": patch_size,
    "n_rotation_classes": n_rotation_classes,
    "n_unlabeled_patches": n_unlabeled_patches,
    "normalization_sample_size": normalization_sample_size,
    "train_fraction": train_fraction,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "max_epochs": max_epochs,
    "early_stopping_patience": early_stopping_patience,
    "gradient_clip_norm": gradient_clip_norm,
    "random_seed": random_seed,
    "random_ce_baseline": random_ce_baseline,
    "random_acc_baseline": random_acc_baseline,
    "raster_dir": str(raster_dir),
    "unlabeled_index_csv": str(unlabeled_index_csv),
}

print("Quality check before training")
print("number of raster files:", len(raster_files))
print("raster shape:", (raster_audit.height, raster_audit.width))
print("patch_size:", patch_size)
print("n_rotation_classes:", n_rotation_classes)
print("number of valid unlabeled patches:", len(rotation_dataset))
print("train/val split sizes:", len(train_dataset), len(val_dataset))
print("batch size:", batch_size)
print("device:", device)
if torch.cuda.is_available():
    print("CUDA GPU name:", torch.cuda.get_device_name(0))
print("model parameter count:", count_trainable_parameters(model))
print("random CE baseline ln(4):", random_ce_baseline)
print("random accuracy baseline:", random_acc_baseline)
print("max_epochs:", max_epochs)
print("learning rate:", learning_rate)
print("encoder checkpoint path:", encoder_best_path)
print("full model checkpoint path:", full_model_best_path)
print("plotting font family and font size:", plot_font_family, 10)

training_log, training_summary = train_rotation_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    full_model_best_path=full_model_best_path,
    encoder_best_path=encoder_best_path,
    last_checkpoint_path=last_checkpoint_path,
    config=config,
    channel_means=channel_means,
    channel_stds=channel_stds,
    max_epochs=max_epochs,
    early_stopping_patience=early_stopping_patience,
    batch_size=batch_size,
    n_rotation_classes=n_rotation_classes,
    grad_clip_norm=gradient_clip_norm,
)
training_log.to_csv(training_log_path, index=False)
print("Training summary:", training_summary)
print("Training log saved:", training_log_path)

Quality check before training
number of raster files: 14
raster shape: (10071, 9596)
patch_size: 32
n_rotation_classes: 4
number of valid unlabeled patches: 20000
train/val split sizes: 18000 2000
batch size: 128
device: cuda
CUDA GPU name: NVIDIA GeForce RTX 4090
model parameter count: 11177220
random CE baseline ln(4): 1.3862943611198906
random accuracy baseline: 0.25
max_epochs: 50
learning rate: 0.0001
encoder checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\rotation\resnet18_rotation_ps32_encoder_best.pt
full model checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\rotation\resnet18_rotation_ps32_full_model_best.pt
plotting font family and font size: Arial 10


epoch 001: train_loss=1.233156, val_loss=0.573568, val_top1=0.7840


epoch 002: train_loss=0.312864, val_loss=0.206274, val_top1=0.9255


epoch 003: train_loss=0.108525, val_loss=0.092325, val_top1=0.9650


epoch 004: train_loss=0.058965, val_loss=0.061483, val_top1=0.9775


...

epoch 044: train_loss=0.002964, val_loss=0.007598, val_top1=0.9980


epoch 045: train_loss=0.002024, val_loss=0.005927, val_top1=0.9970


epoch 046: train_loss=0.000806, val_loss=0.004211, val_top1=0.9975


epoch 047: train_loss=0.001314, val_loss=0.017666, val_top1=0.9945


Training summary: {'best_epoch': 37, 'best_val_loss': 0.004128127808042336, 'best_train_loss': 0.0028211253293751117, 'best_val_acc_top1': 0.9985}
Training log saved: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\SSL_rotation_ps32\training_logs\rotation_ps32_training_log.csv


## 12. Save best pretrained encoder and logs

In [12]:
required_log_columns = ["epoch", "train_loss", "val_loss", "train_acc_top1", "val_acc_top1", "learning_rate", "batch_size", "n_rotation_classes"]
missing_log_columns = [column for column in required_log_columns if column not in training_log.columns]
if missing_log_columns:
    raise ValueError(f"Training log missing columns: {missing_log_columns}")
for path in [training_log_path, encoder_best_path, full_model_best_path, last_checkpoint_path]:
    if not path.exists():
        raise FileNotFoundError(path)
encoder_checkpoint = torch.load(encoder_best_path, map_location="cpu")
full_checkpoint = torch.load(full_model_best_path, map_location="cpu")
required_encoder_keys = {"encoder_state_dict", "epoch", "val_loss", "val_acc_top1", "config", "channel_means", "channel_stds"}
missing_encoder_keys = required_encoder_keys - set(encoder_checkpoint.keys())
if missing_encoder_keys:
    raise KeyError(f"Encoder checkpoint missing keys: {sorted(missing_encoder_keys)}")
print("Encoder checkpoint keys:", sorted(encoder_checkpoint.keys()))
print("Full model checkpoint keys:", sorted(full_checkpoint.keys()))
print("Encoder checkpoint epoch:", encoder_checkpoint["epoch"])
print("Encoder checkpoint val_loss:", encoder_checkpoint["val_loss"])
print("Encoder checkpoint val_acc_top1:", encoder_checkpoint["val_acc_top1"])
print("Encoder state keys:", len(encoder_checkpoint["encoder_state_dict"]))

Encoder checkpoint keys: ['channel_means', 'channel_stds', 'config', 'encoder_state_dict', 'epoch', 'val_acc_top1', 'val_loss']
Full model checkpoint keys: ['channel_means', 'channel_stds', 'config', 'epoch', 'model_state_dict', 'optimizer_state_dict', 'train_loss', 'val_acc_top1', 'val_loss']
Encoder checkpoint epoch: 37
Encoder checkpoint val_loss: 0.004128127808042336
Encoder checkpoint val_acc_top1: 0.9985
Encoder state keys: 120


## 13. Visualize rotation examples and training curves

In [13]:
plot_rotation_training_curves(training_log, loss_acc_curve_png, random_loss=random_ce_baseline, random_accuracy=random_acc_baseline)
plot_rotation_training_curves(training_log, loss_acc_curve_pdf, random_loss=random_ce_baseline, random_accuracy=random_acc_baseline)

example_X = rotation_dataset.read_normalized_patch(0).unsqueeze(0)
rotated_examples = [apply_rotation(example_X[0], rotation_class).unsqueeze(0) for rotation_class in range(4)]
plot_rotation_examples(
    example_X,
    rotated_examples,
    output_path=rotation_example_png,
    channel_indices=[0, 1, 2],
    channel_names=["factor_01", "factor_02", "factor_03"],
    rotation_labels=ROTATION_CLASS_LABELS,
)
plot_rotation_examples(
    example_X,
    rotated_examples,
    output_path=rotation_example_pdf,
    channel_indices=[0, 1, 2],
    channel_names=["factor_01", "factor_02", "factor_03"],
    rotation_labels=ROTATION_CLASS_LABELS,
)
for path in [loss_acc_curve_png, loss_acc_curve_pdf, rotation_example_png, rotation_example_pdf]:
    if not path.exists():
        raise FileNotFoundError(path)
    print("Saved figure:", path)

Saved figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_rotation_ps32\training_curves\rotation_ps32_loss_accuracy_curve.png
Saved figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_rotation_ps32\training_curves\rotation_ps32_loss_accuracy_curve.pdf
Saved figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_rotation_ps32\rotation_examples\rotation_examples_ps32.png
Saved figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_rotation_ps32\rotation_examples\rotation_examples_ps32.pdf


## 14. Plot validation confusion matrix

In [14]:
best_full = torch.load(full_model_best_path, map_location=device)
model.load_state_dict(best_full["model_state_dict"])
val_pred_result = evaluate_rotation(model, val_loader, criterion, device, return_predictions=True)
plot_rotation_confusion_matrix(val_pred_result["y_true"], val_pred_result["y_pred"], confusion_png, class_labels=ROTATION_CLASS_LABELS)
plot_rotation_confusion_matrix(val_pred_result["y_true"], val_pred_result["y_pred"], confusion_pdf, class_labels=ROTATION_CLASS_LABELS)
for path in [confusion_png, confusion_pdf]:
    if not path.exists():
        raise FileNotFoundError(path)
    print("Saved confusion figure:", path)

Saved confusion figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_rotation_ps32\confusion\rotation_val_confusion_ps32.png
Saved confusion figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_rotation_ps32\confusion\rotation_val_confusion_ps32.pdf


## 15. Print final summary and next-step notes

In [15]:
final_train_loss = float(training_log["train_loss"].iloc[-1])
final_val_loss = float(training_log["val_loss"].iloc[-1])
final_train_acc = float(training_log["train_acc_top1"].iloc[-1])
final_val_acc = float(training_log["val_acc_top1"].iloc[-1])
best_val_acc = float(training_summary["best_val_acc_top1"])
print("Rotation SSL pretraining summary")
print("number of valid unlabeled patches:", len(rotation_dataset))
print("best epoch:", training_summary["best_epoch"])
print("best validation loss:", training_summary["best_val_loss"])
print("best validation top-1 accuracy:", best_val_acc)
print("final train loss:", final_train_loss)
print("final val loss:", final_val_loss)
print("final train accuracy:", final_train_acc)
print("final validation accuracy:", final_val_acc)
print("encoder checkpoint path:", encoder_best_path)
print("full model checkpoint path:", full_model_best_path)
print("last checkpoint path:", last_checkpoint_path)
print("training log path:", training_log_path)
print("training curve figure path:", loss_acc_curve_png)
print("rotation example figure path:", rotation_example_png)
print("confusion figure path:", confusion_png)
print("Confirmed: no landslide labels were used, no supervised fine-tuning was performed, and no susceptibility maps were generated.")
if best_val_acc > random_acc_baseline + 0.05:
    print("Rotation prediction learned a non-random geometric representation.")
elif abs(best_val_acc - random_acc_baseline) <= 0.05:
    print("Rotation prediction remained near the random baseline and should be interpreted as a weak geometric pretext task.")
else:
    print("Rotation prediction failed to learn the pretext task and should be treated as an unsuitable SSL pretext task for this dataset.")
print("Next step: use the Rotation encoder checkpoint in supervised SCV fine-tuning notebook: notebooks/14_finetune_rotation_resnet18_ps32_scv.ipynb")
rotation_dataset.close()

Rotation SSL pretraining summary
number of valid unlabeled patches: 20000
best epoch: 37
best validation loss: 0.004128127808042336
best validation top-1 accuracy: 0.9985
final train loss: 0.001313669465763572
final val loss: 0.01766621629358269
final train accuracy: 0.9995
final validation accuracy: 0.9945
encoder checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\rotation\resnet18_rotation_ps32_encoder_best.pt
full model checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\rotation\resnet18_rotation_ps32_full_model_best.pt
last checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\rotation\resnet18_rotation_ps32_last.pt
training log path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\SSL_rotation_ps32\training_logs\rotation_ps32_training_log.csv
training curve figure path: D:\SBU first year\First year paper\LSM